# Notebook 08: Blog Asset Generation

This notebook generates all key figures for the quantum foundations blog post.
Every figure is produced deterministically and saved to `assets/blog/` as PNG
files using the `save_figure` utility.

Figures generated:
1. `probability-vs-amplitude.png` -- Classical vs quantum comparison panel
2. `same-probabilities-different-states.png` -- Demo A: same probs, different amplitudes
3. `interference-cancellation.png` -- Constructive vs destructive interference
4. `oracle-phase-demo.png` -- Demo C: oracle phase flip + diffusion
5. `grover-trajectory.png` -- Target probability across Grover iterations
6. `grover-plane.png` -- 2D rotation geometry of Grover's algorithm
7. `qubit-geometry.png` -- Unit circle plot of a qubit state

In [ ]:
import sys
sys.path.insert(0, '../src')

import numpy as np
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for deterministic export
import matplotlib.pyplot as plt

from quantum_demo.linalg import ket
from quantum_demo.states import (
    qubit_state, equal_superposition, amplitudes_to_probabilities, pretty_basis_labels
)
from quantum_demo.gates import H, Z, apply_gate, phase_oracle, diffusion_operator
from quantum_demo.grover import (
    grover_run, grover_optimal_iterations,
    target_probability_trajectory, reduced_grover_plane_coordinates
)
from quantum_demo.interference import hadamard_interference_demo
from quantum_demo.viz.static_plots import (
    plot_probabilities, plot_real_amplitudes,
    plot_qubit_state_2d, plot_classical_vs_quantum_panel
)
from quantum_demo.viz.grover_plots import plot_grover_trajectory, plot_grover_plane
from quantum_demo.viz.state_plots import plot_state_comparison, plot_oracle_phase_demo
from quantum_demo.viz.export import save_figure

# Output directory (relative to notebook location)
ASSET_DIR = '../assets/blog'

# Track all exported files
exported_files = []

## Figure 1: Probability vs Amplitude (Classical vs Quantum)

A side-by-side comparison showing how classical probability distributions
differ from quantum states.  Classical systems have only non-negative
probabilities; quantum systems have signed (or complex) amplitudes that
square to give probabilities.

In [ ]:
# Classical: uniform distribution over 4 outcomes
classical_probs = np.array([0.25, 0.25, 0.25, 0.25])

# Quantum: non-uniform amplitudes that give the same marginal probabilities
# but with sign structure
quantum_state = np.array([0.5, 0.5, -0.5, 0.5], dtype=np.complex128)

labels = ['0', '1', '2', '3']

fig = plot_classical_vs_quantum_panel(
    classical_probs, quantum_state,
    labels=labels,
    title="Classical Probabilities vs Quantum Amplitudes"
)

paths = save_figure(fig, f'{ASSET_DIR}/probability-vs-amplitude', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Figure 2: Same Probabilities, Different States (Demo A)

Two quantum states that yield identical measurement probabilities but have
fundamentally different amplitude structures.  This demonstrates that
amplitudes carry more information than probabilities alone.

In [ ]:
# |+> and |-> have identical probabilities but opposite sign on |1>
ket0 = ket(0, 2)
plus = apply_gate(ket0, H)   # (1/sqrt2)(|0> + |1>)
minus = apply_gate(plus, Z)  # (1/sqrt2)(|0> - |1>)

fig = plot_state_comparison(
    states=[plus, minus],
    state_labels=['|+> = (|0>+|1>)/sqrt(2)', '|-> = (|0>-|1>)/sqrt(2)'],
    basis_labels=['|0>', '|1>'],
    title='Demo A: Same Probabilities, Different States'
)

paths = save_figure(fig, f'{ASSET_DIR}/same-probabilities-different-states', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Figure 3: Interference Cancellation

Constructive interference (H applied to |+>, amplitudes reinforce) vs
destructive interference (H applied to |->, amplitudes cancel).  This shows
how the Hadamard gate converts phase differences into measurable outcomes.

In [ ]:
# Use the interference demo module
demo = hadamard_interference_demo()

# H|0> = |+> --H--> |0>  (constructive: both paths to |0> add)
# H|0> = |+> --Z--> |-> --H--> |1>  (destructive: paths to |0> cancel)
constructive = demo['HH_ket0']    # HH|0> = |0>
destructive = demo['HZH_ket0']    # HZH|0> = |1>
superposition = demo['H_ket0']    # H|0> = |+>

fig = plot_state_comparison(
    states=[superposition, constructive, destructive],
    state_labels=[
        'H|0> = |+>\n(superposition)',
        'HH|0> = |0>\n(constructive)',
        'HZH|0> = |1>\n(destructive)'
    ],
    basis_labels=['|0>', '|1>'],
    title='Interference: Phase Determines Outcome'
)

paths = save_figure(fig, f'{ASSET_DIR}/interference-cancellation', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Figure 4: Oracle Phase Demo (Demo C)

The oracle flips the sign of the target amplitude but leaves all probabilities
unchanged.  Only after the diffusion operator does the target probability
increase.  This is the core mechanism of Grover's algorithm.

In [ ]:
# 8 states (3 qubits), target at index 3
N = 8
target_idx = 3
labels_3q = pretty_basis_labels(3)

before = equal_superposition(N)
oracle = phase_oracle(target_idx, N)
after_oracle = apply_gate(before, oracle)
diffusion = diffusion_operator(N)
after_diffusion = apply_gate(after_oracle, diffusion)

fig = plot_oracle_phase_demo(
    before_oracle=before,
    after_oracle=after_oracle,
    after_diffusion=after_diffusion,
    target_index=target_idx,
    labels=labels_3q,
    title='Demo C: Oracle Phase Flip Is Invisible -- Until Diffusion'
)

paths = save_figure(fig, f'{ASSET_DIR}/oracle-phase-demo', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Figure 5: Grover Trajectory

Target probability as a function of Grover iteration count.  Shows the
sinusoidal oscillation and the optimal stopping point.

In [ ]:
N_grover = 64  # 6 qubits
target_grover = 42
k_opt = grover_optimal_iterations(N_grover)
n_show = k_opt + 8  # show past the optimum to see the oscillation

probs_traj = target_probability_trajectory(N_grover, target_grover, n_show)

fig = plot_grover_trajectory(
    probs_traj,
    title=f'Grover Search: P(target) vs Iteration (N={N_grover}, target={target_grover})'
)

paths = save_figure(fig, f'{ASSET_DIR}/grover-trajectory', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Figure 6: Grover Plane (2D Rotation Geometry)

The state trajectory projected onto the 2D plane spanned by |target> and
|s_perp>.  Each Grover iteration is a fixed-angle rotation in this plane.

In [ ]:
# Use N=16 for a clear visualization (fewer iterations, wider rotation angle)
N_plane = 16
target_plane = 7
k_opt_plane = grover_optimal_iterations(N_plane)

traj_plane = grover_run(N_plane, target_plane, k_opt_plane + 2)
coords = [reduced_grover_plane_coordinates(s, target_plane) for s in traj_plane]

fig = plot_grover_plane(
    coords,
    title=f'Grover Geometry: State Rotation in 2D Plane (N={N_plane})'
)

paths = save_figure(fig, f'{ASSET_DIR}/grover-plane', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Figure 7: Qubit Geometry (Unit Circle)

A single-qubit state visualized on the unit circle, showing the amplitude
components along |0> and |1> and the resulting measurement probabilities.

In [ ]:
# A qubit state at an interesting angle
psi = qubit_state(np.sqrt(3), 1)  # normalized: [sqrt(3)/2, 1/2]

fig = plot_qubit_state_2d(
    psi,
    title='Qubit State on the Unit Circle'
)

paths = save_figure(fig, f'{ASSET_DIR}/qubit-geometry', dpi=150)
exported_files.extend(paths)
plt.close(fig)
print(f"Saved: {paths}")

## Summary of exported files

In [ ]:
print("=" * 60)
print("BLOG ASSET GENERATION COMPLETE")
print("=" * 60)
print(f"\nTotal files exported: {len(exported_files)}")
print(f"Output directory: {ASSET_DIR}/")
print()
for i, path in enumerate(exported_files, 1):
    print(f"  {i}. {path}")

print("\nAll figures generated deterministically.")
print("Ready for blog integration.")